# Supply Chain Maze

In [31]:
# imports
import cv2
import numpy as np
# pip install Pillow
    # open Anaconda Prompt and paste above line (without '#') to install package
from PIL import Image

In [32]:
# get_limits function
# function that gives a range on hues given a color
def get_limits(color):
    c = np.uint8([[color]])
    hsvC = cv2.cvtColor(c, cv2.COLOR_BGR2HSV)
    
    lowerLimit = hsvC[0][0][0] - 10, 100, 100
    upperLimit = hsvC[0][0][0] + 10, 255, 255
    # the +/-10 defines the range of hues that fall within the limits (the h in hsv)
    # the range on saturation and value is much bigger because we are only looking for hue
    
    lowerLimit = np.array(lowerLimit, dtype=np.uint8)
    upperLimit = np.array(upperLimit, dtype=np.uint8)

    return lowerLimit, upperLimit

In [ ]:
# run_calibration function

# Call by pressing [c] from the opening menu
# Records new points for the maze and saves them in calibration_output.txt

def run_calibration(VCInput, color, matrix_length):
    """Calibrate node positions using a color marker.
    Pass the existing capture object so calibration runs in the same window.
    Place marker on each node and press [spacebar] to record.
    Press [backspace] to undo the last point.
    Press [ESC] to cancel (keeps existing points unchanged).
    Returns list of 9 [x, y] points, or [] if cancelled."""
    detection_size = 100  # how many pixels in a group needed to detect an object (may be different from the normal program detection_size)

    # Use the provided capture, or create a new one if none was passed
    capture = cv2.VideoCapture(VCInput)

    count = 0
    nodes = list()
    key = 0
    while True:
        ret, frame = capture.read()
        if flip_camera:    # jmhere
            frame = cv2.flip(frame, 1)
        if frame.shape[1] / frame.shape[0] == 4 / 3:  # crops the video feed if it has 4:3 aspect ratio (so it doesn't stretch during resize)
            trim = int(frame.shape[0] * 0.125)  # trims 12.5% from each side of height to change into 16:9 aspect ratio
            frame = frame[trim:frame.shape[0] - trim, :]
        frame = cv2.resize(frame, (1280, 720), interpolation=cv2.INTER_AREA)

        shape = frame.shape
        height = shape[0]
        width = shape[1]
        cqmh = height / 480
        cqmw = width / 640

        frame_blur = cv2.GaussianBlur(frame, (11, 11), 9)
        frame_hsv = cv2.cvtColor(frame_blur, cv2.COLOR_BGR2HSV)
        lowerLimit, upperLimit = get_limits(color)

        mask = cv2.inRange(frame_hsv, lowerLimit, upperLimit)
        contours, hierarchy = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        bboxes = list()
        for cnt in contours:
            if cv2.contourArea(cnt) > detection_size * cqmh * cqmw:
                x1, y1, w, h = cv2.boundingRect(cnt)
                c = list([int(x1 + w/2), int(y1 + h/2)])
                newBox = list([c[0], c[1], x1, y1, w, h, cv2.contourArea(cnt)])
                for i in bboxes[:]:
                    cxi, cyi, x1i, y1i, wi, hi, si = i
                    if np.sqrt((c[0] - cxi)**2 + (c[1] - cyi)**2) < np.sqrt(w**2 + h**2)/3 + np.sqrt(wi**2 + hi**2)/3 + cqmw*30:
                        bboxes.remove(i)
                        newBox[2], newBox[3] = min(x1, x1i), min(y1, y1i)
                        newBox[4], newBox[5] = max(x1+w, x1i+wi) - newBox[2], max(y1+h, y1i+hi) - newBox[3]
                        newBox[0], newBox[1] = int(newBox[2] + newBox[4]/2), int(newBox[3] + newBox[5]/2)
                bboxes.append(newBox)

        maxVal = 0
        point = 0
        for i in bboxes:
            cx, cy, x1, y1, w, h, s = i
            if s > maxVal:
                maxVal = s
                point = i

        if point != 0:
            cx, cy, x1, y1, w, h, s = point
            cv2.rectangle(frame, (x1, y1), (x1 + w, y1 + h), (0, 0, 255), 2)
            cv2.line(frame, (cx, cy), (cx, cy), (255, 0, 0), 15)
            cv2.line(frame, (cx, cy), (cx, cy), (0, 255, 255), 10)
            cv2.line(frame, (cx, cy), (cx, cy), (0, 0, 255), 5)

        text_color = (150, 150, 150)
        text_border_color = (0, 0, 0)
        if count == 0:
            label = 'Calibrating: Start'
        elif count == matrix_length:
            label = 'Calibrating: End'
        else:
            label = f'Calibrating: Point {count}'
        # show on screen which point is being calibrated
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            cv2.putText(frame, label, (25 + 2*dx, int(height - 115) + 2*dy), cv2.FONT_HERSHEY_TRIPLEX, 1.3, text_color, 2)
        cv2.putText(frame, label, (25, int(height - 115)), cv2.FONT_HERSHEY_TRIPLEX, 1.3, text_border_color, 2)
        # show on screen what to press on keyboard
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            cv2.putText(frame, 'Press [spacebar] to Record Point', (25 + dx, int(height - 70) + dy), cv2.FONT_HERSHEY_TRIPLEX, 1.1, text_border_color, 2)
        cv2.putText(frame, 'Press [spacebar] to Record Point', (25, int(height - 70)), cv2.FONT_HERSHEY_TRIPLEX, 1.1, text_color, 2)
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            cv2.putText(frame, 'Press [backspace] to Undo', (25 + dx, int(height - 25) + dy), cv2.FONT_HERSHEY_TRIPLEX, 1.1, text_border_color, 2)
        cv2.putText(frame, 'Press [backspace] to Undo', (25, int(height - 25)), cv2.FONT_HERSHEY_TRIPLEX, 1.1, text_color, 2)

        if key == 32 and point != 0:   # spacebar - record point
            nodes.append([point[0], point[1]])
            point = 0
            count += 1
            if count >= matrix_length + 1:
                break
            cv2.waitKey(250)

        if key == 8 and len(nodes) > 0:   # backspace - undo last point
            nodes.pop()
            count -= 1
            cv2.waitKey(250)

        cv2.imshow('Supply Chain Maze', frame)
        key = cv2.waitKey(10) & 0xFF
        if key == 27:   # ESC - cancel, return empty list so points stay unchanged
            nodes = []
            break

    # Only release the capture if we created it ourselves (i.e., none was passed in)
    capture.release()
    cv2.destroyAllWindows()
    if nodes:
        print('Calibration complete:', nodes)
        with open("calibration_output.txt", "w") as f:  # write the output to calibration_output.txt so it can be used again if the program is terminated
            for item in nodes:
                f.write(f"{item[0]},{item[1]}\n")
    return nodes

In [ ]:
# adjustable inputs/parameters

# default points
default_points = [[264, 131], [275, 359], [270, 522], [432, 127], [433, 237], [426, 403], [428, 523], [638, 102], [644, 241], [642, 389], [654, 523], [811, 102], [811, 325], [822, 493]]
use_default_points = False  # usually False, decides if we use the default points or the points from the previous calibration output

#color = [85, 145, 43]
color = [202, 137, 65]                # color in BGR colorspace
line_color = [0, 255, 255]            # color used to draw path
line_size = 4                         # line thickness
line_color_check = [0, 0, 255]        # line color used to draw the correct solution
detection_size = 500                  # minimum size of detection required to mark color as an object
detection_range = 10                  # determines how close object needs to be to points to see an overlap (higher number -> easier to connect points)
flip_camera = False   # jmhere        # do we horizontally flip camera? flipping is good for mirroring mvoements with laptop carmera, not flipping is good for forward-facing external camera

VCInput = 2                            # used in cv2.VideoCapture to select a camera to use (usually 0 or 1, could be 2, 3, 4, etc. if computer has many camera inputs)
capture = cv2.VideoCapture(VCInput)    # VCInput sets camera and is used for all instances of VideoCapture to keep camera consistent

In [ ]:
# main program

                     # node   s   1   2   3   4   5   6   7   8   9  10  11  12   e
adjacency_matrix = np.array([[0,  8,  0, 11,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],   # start
                             [8,  0,  2,  5,  0,  0,  5,  0,  0,  0,  0,  0,  0,  0],   # 1
                             [0,  2,  0,  0,  0,  0,  5,  0,  0,  0,  0,  0,  0,  0],   # 2
                             [11, 5,  0,  0, 14,  0,  0,  0,  0,  0,  0,  0,  0,  0],   # 3
                             [0,  0,  0, 14,  0,  2,  0,  0,  5,  0,  0,  0,  0,  0],   # 4
                             [0,  0,  0,  0,  2,  0,  8,  0, 11, 14,  0,  0,  0,  0],   # 5
                             [0,  5,  5,  0,  0,  8,  0,  0,  0,  0,  0,  0,  0,  0],   # 6
                             [0,  0,  0,  0,  0,  0,  0,  0,  2,  0,  0,  8, 14,  0],   # 7
                             [0,  0,  0,  0,  5, 11,  0,  2,  0,  5,  0,  0,  0,  0],   # 8
                             [0,  0,  0,  0,  0, 14,  0,  0,  5,  0,  8,  0,  0,  0],   # 9
                             [0,  0,  0,  0,  0,  0,  0,  0,  0,  8,  0,  0,  8, 11],   # 10
                             [0,  0,  0,  0,  0,  0,  0,  8,  0,  0,  0,  0,  5,  0],   # 11
                             [0,  0,  0,  0,  0,  0,  0, 14,  0,  0,  8,  5,  0,  2],   # 12
                             [0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 11,  0,  2,  0]])  # end
                # non-zero values indicate that two nodes (corresponing to the row and column) are adjacent
                # non-zero values describe the cost of taking a path between two nodes
                # the matrix is symmetric, so it doesn't matter if you take [3][4] or [4][3] ; they tell you the same thing
matrix_length = len(adjacency_matrix) - 1  # number of nodes in graph minus 1 (zero-indexed)

# use the points from calibration_output.txt (if they are valid points)
if use_default_points == False:
    try:  # checks for errors so errors won't stop the whole program
        with open("calibration_output.txt", "r") as f:  # "r" read calibration_output
            cal_output = []
            for line in f:
                parts = line.strip().split(",")
                cal_output.append([int(parts[0]), int(parts[1])])
        
        # check to make sure that the tuple from calibraiton_output is the right size
            # checking individual tuple length = 2 and full list length is the same as the matrix
        if all(len(t) == 2 for t in cal_output) and (len(cal_output) == matrix_length + 1):
            points = cal_output  # use the points from calibration_output if it passes inspection
        else:
            print("wrong size for calibration_output.txt (default_points used)")
            points = default_points  # use default_points if cal_output is the wrong size
    except Exception:  # if any error is found, use default_points instead of calibration output
        print("improper format for calibration_output.txt (default_points used)")
        points = default_points
else:
    points = default_points  # use default_points if use_default_points == True

while len(points) != matrix_length + 1:  # adjust size of points array if it's the wrong size for the given adjacency matrix
    if len(points) < matrix_length + 1: points.append([640, 390])  # add filler point if there are too few
    if len(points) > matrix_length + 1: points.pop()  # remove last point if there are too many

mode = 0

if (line_color == [0, 0, 0]): # breaks if you use black ([0, 0, 0]), so we adjust it
    line_color = [1, 1, 1]
if (line_color_check == [0, 0, 0]): # breaks if you use black ([0, 0, 0]), so we adjust it
    line_color_check = [1, 1, 1]

# big while loop, loops for each instance of the optimization problems
while True:
    first = False
    count = 0
    prev_node = -1
    current_node = -1
    path_value = 0
    nodes_taken = np.zeros(matrix_length + 1)
    nodes_taken_ordered = []
    complete = False
    path = 0
    text = 0

    # loops for each frame of the video feed
    while True:
        ret, frame = capture.read()
        if flip_camera:    # jmhere
            frame = cv2.flip(frame, 1)
        if frame.shape[1] / frame.shape[0] == 4 / 3:  # crops the video feed if it has 4:3 aspect ratio (so it doesn't stretch during resize)
            trim = int(frame.shape[0] * 0.125)  # trims 12.5% from each side of height to change into 16:9 aspect ratio
            frame = frame[trim:frame.shape[0] - trim, :]
        frame = cv2.resize(frame, (1280, 720), interpolation=cv2.INTER_AREA)
        
        if (first == False): # runs only once
            # get height and width of video
            shape = frame.shape
            height = shape[0]
            width = shape[1]
            cqm = width / 640  # scales detection with video quality (because it depends on number of pixels)
            line_thickness = int(line_size * cqm)
            # create path image to draw on (based on shape of webcam feed)
            path = np.array(Image.new("RGB", (width, height), (0,0,0)))
            # draw dots to mark each node at each node
            for i, v in enumerate(points):    # draws markers at every point
                cv2.circle(path, v, 1, (1, 1, 1), int(9*cqm))
                if i == 0 or i == matrix_length:
                    cv2.circle(path, v, 1, [202, 137, 65], int(6*cqm))
                else:
                    cv2.circle(path, v, 1, line_color, int(6*cqm))                
                cv2.circle(path, v, 1, (1, 1, 1), int(3*cqm))
            # draw numbers between paths
            text = np.array(Image.new("RGB", (width, height), (0,0,0)))
            for i, val in enumerate(adjacency_matrix):
                for j in range(i):
                    if adjacency_matrix[i][j] != 0:
                        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                            cv2.putText(text, f'{adjacency_matrix[i][j]}', (int((points[i][0] + points[j][0])/2) + dx, int((points[i][1] + points[j][1])/2) + dy), cv2.FONT_HERSHEY_SIMPLEX, 0.5*cqm, (255, 255, 255), int(2*cqm))
                        cv2.putText(text, f'{adjacency_matrix[i][j]}', (int((points[i][0] + points[j][0])/2), int((points[i][1] + points[j][1])/2)), cv2.FONT_HERSHEY_SIMPLEX, 0.5*cqm, (1, 1, 1), int(2*cqm))
            # set first to True so this block doesn't run again
            first = True

        # draw instruction text
        if mode != 0:
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                cv2.putText(frame, 'Press [backspace] to Undo', (int(25*cqm) + dx, int(height - 12.5*cqm) + dy), cv2.FONT_HERSHEY_TRIPLEX, 0.55*cqm, (0, 0, 0), int(1*cqm))
            cv2.putText(frame, 'Press [backspace] to Undo', (int(25*cqm), int(height - 12.5*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 0.55*cqm, (150, 150, 150), int(1*cqm))

        # key input to decide if we are doing Shortest Path Problem, Longest Path Problem, or Calibration
        if mode == 0:  # Shows the options to choose
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:  # places the text 4 times in slightly different spots to outline text
                cv2.putText(frame, 'Press [C] for Calibration', (int(25*cqm) + dx, int(height - 82.5*cqm) + dy), cv2.FONT_HERSHEY_TRIPLEX, 0.8*cqm, (0, 0, 0), int(1*cqm))
                cv2.putText(frame, 'Press [S] for Shortest Path Problem', (int(25*cqm) + dx, int(height - 47.5*cqm) + dy), cv2.FONT_HERSHEY_TRIPLEX, 0.8*cqm, (0, 0, 0), int(1*cqm))
                cv2.putText(frame, 'Press [L] for Longest Path Problem', (int(25*cqm) + dx, int(height - 12.5*cqm) + dy), cv2.FONT_HERSHEY_TRIPLEX, 0.8*cqm, (0, 0, 0), int(1*cqm))
            cv2.putText(frame, 'Press [C] for Calibration', (int(25*cqm), int(height - 82.5*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 0.8*cqm, (150, 150, 150), int(1*cqm))
            cv2.putText(frame, 'Press [S] for Shortest Path Problem', (int(25*cqm), int(height - 47.5*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 0.8*cqm, (150, 150, 150), int(1*cqm))
            cv2.putText(frame, 'Press [L] for Longest Path Problem', (int(25*cqm), int(height - 12.5*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 0.8*cqm, (150, 150, 150), int(1*cqm))
            key = cv2.waitKey(10) & 0xFF
            if key == 115 or key == 83:              # 115 is ASCII for [s], 83 for [S]
                mode = 2      # choose Shortest Path Problem
            if key == 108 or key == 76:              # 116 is ASCII for [l], 76 for [L]
                mode = 3      # choose Longest Path
            if key == 99 or key == 67:               #  99 is ASCII for [c], 67 for [C]
                mode = 1      # choose Calibration
                capture.release()                        # free camera so calibration can use it
                new_points = run_calibration(VCInput, color, matrix_length)           # run calibration; returns [] if cancelled
                capture = cv2.VideoCapture(0)            # re-open camera for main program
                mode = 0
                if new_points:
                    points = new_points                  # update points only if calibration completed
                break
            if key == 27:               # 27 is ASCII for [ESC]
                break                   # HOLD [ESC] to end program
            else:
                cv2.imshow('Supply Chain Maze', frame)
                continue  # skips everything onward if we haven't decided which option to run
    
        frame_blur = cv2.GaussianBlur(frame, (11, 11), 9) # blurring the image may help get the desired result, but it can be removed
        
        frame_hsv = cv2.cvtColor(frame_blur, cv2.COLOR_BGR2HSV) # convert to HSV
        lowerLimit, upperLimit = get_limits(color) # range of hues that we want the software to detect
        
        mask = cv2.inRange(frame_hsv, lowerLimit, upperLimit) # detects objects in color range
        contours, hierarchy = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        bboxes = list()
        for cnt in contours:
            if cv2.contourArea(cnt) > cqm**2*detection_size:  # only continues if size of the object is large enough (removes noise)
                x1, y1, w, h = cv2.boundingRect(cnt)  # finds a bounding box for each object
                c = list([int(x1 + w/2), int(y1 + h/2)])  # centerpoint of bbox
                # check other bboxes to see if we want to combine them into 1 box
                newBox = list([c[0], c[1], x1, y1, w, h, cv2.contourArea(cnt)])
                for i in bboxes[:]:
                    cxi, cyi, x1i, y1i, wi, hi, si = i  # bbox we check the newBox against
                    if np.sqrt((c[0] - cxi)**2 + (c[1] - cyi)**2) < np.sqrt(w**2 + h**2)/3 + np.sqrt(wi**2 + hi**2)/3 + cqm*25: # if centerpoints are close enough (scales with box size)
                        bboxes.remove(i)
                        # reassign bbox boundaries so the new box contains both nearby boxes
                        newBox[2], newBox[3] = min(x1, x1i), min(y1, y1i)  # reassigns x1 and y1 values
                        newBox[4], newBox[5] = max(x1+w, x1i+wi) - newBox[2], max(y1+h, y1i+hi) - newBox[3]  # reassgins w and h values
                        newBox[0], newBox[1] = int(newBox[2] + newBox[4]/2), int(newBox[3] + newBox[5]/2)  # reassigns centerpoint values
                bboxes.append(newBox)
    
        # overlay the path over the webcam feed
        mask2 = cv2.cvtColor(path, cv2.COLOR_BGR2GRAY)
        mask2 = cv2.threshold(mask2, 0, 255, cv2.THRESH_BINARY)[1]
        mask2inv = cv2.bitwise_not(mask2)
        pathfg = cv2.bitwise_and(path, path, mask=mask2) # extract foreground
        framebg = cv2.bitwise_and(frame, frame, mask=mask2inv) # extract background
        frame = cv2.add(framebg, pathfg) # combine foreground and background
        # overlay the numbers between nodes
        mask2 = cv2.cvtColor(text, cv2.COLOR_BGR2GRAY)
        mask2 = cv2.threshold(mask2, 0, 255, cv2.THRESH_BINARY)[1]
        mask2inv = cv2.bitwise_not(mask2)
        pathfg = cv2.bitwise_and(text, text, mask=mask2) # extract foreground
        framebg = cv2.bitwise_and(frame, frame, mask=mask2inv) # extract background
        frame = cv2.add(framebg, pathfg) # combine foreground and background
        
        # find the largest object
        maxVal = 0
        marker = 0
        for i in bboxes:  
            cx, cy, x1, y1, w, h, s = i
            if s > maxVal:
                maxVal = s
                marker = i
    
        # draw circle around current node
        if current_node != -1:
            cv2.circle(frame, points[current_node], int(20*cqm), (0, 0, 255), int(3*cqm))
            
        # draw bounding box
        if marker != 0:
            cx, cy, x1, y1, w, h, s = marker
            cv2.rectangle(frame, (x1, y1), (x1 + w, y1 + h), (0, 0, 255), 2)
            cv2.line(frame, (cx, cy), (cx, cy), (255, 0, 0), int(15*cqm))
            cv2.line(frame, (cx, cy), (cx, cy), (0, 255, 255), int(10*cqm))
            cv2.line(frame, (cx, cy), (cx, cy), (0, 0, 255), int(5*cqm))
    
            # check if we are close enough to a node
            for index, value in enumerate(points):
                # sets node to "start" if it gets close enough to the "start" node
                if current_node == -1 and index == 0 and (marker[0]-value[0])**2+(marker[1]-value[1])**2 < cqm*50*detection_range:
                    current_node = 0
                    nodes_taken[index] = 1
                    nodes_taken_ordered.append(index)
                # sets new node if close enough to an adjacent node and if node is not the previous node
                elif (current_node != -1) and (adjacency_matrix[current_node][index] != 0) and (index != prev_node) and ((marker[0]-value[0])**2+(marker[1]-value[1])**2 < cqm*50*detection_range):
                    if mode == 2:  # shortest path
                        cv2.line(path, (points[current_node][0], points[current_node][1]), (points[index][0], points[index][1]), line_color, line_thickness)
                        nodes_taken[index] = 1
                        nodes_taken_ordered.append(index)
                        prev_node = current_node
                        current_node = index
                        path_value += adjacency_matrix[prev_node][current_node]
                    elif (mode == 3) and (nodes_taken[index] == 0):  # longest path, only go if node has never been visited
                        cv2.line(path, (points[current_node][0], points[current_node][1]), (points[index][0], points[index][1]), line_color, line_thickness)
                        nodes_taken[index] = 1
                        nodes_taken_ordered.append(index)
                        prev_node = current_node
                        current_node = index
                        path_value += adjacency_matrix[prev_node][current_node]
                    if index == matrix_length:  # check that they got to the end and mark complete
                        complete = -1

        
        #cv2.imshow('blur', frame_blur)
        #cv2.imshow('mask', mask)
        cv2.imshow('Supply Chain Maze', frame)
        
        if complete == True:
            cv2.waitKey(1000)
            break
    
        if complete == -1:
            complete = True
        
        key = cv2.waitKey(10) & 0xFF

        if key == 27:               # 27 is ASCII for [ESC]
            break                   # ends loop when [ESC] is pressed

        if key == 32:               # 32 is ASCII for [spacebar]
            mode = 0
            break                   # ends loop when [spacebar] is pressed

        if key == 8 and len(nodes_taken_ordered) > 1:   # 27 is ASCII for [backspace]
            path_value -= adjacency_matrix[nodes_taken_ordered[-1]][nodes_taken_ordered[-2]]
            current_node = nodes_taken_ordered[-2]
            if current_node == 0:
                previous_node = -1
            else:
                previous_node = nodes_taken_ordered[-3]
            nodes_taken[nodes_taken_ordered[-1]] = 0   # doesn't work correctly if you visited a node more than once, but Longest Path doesn't allow for that, and Shortest Path doesn't have any functionality with it
            nodes_taken_ordered.pop()                   # removes the last node when [backspace] is pressed and updates score

            # remake the "path" variable to erase the paths taken visual on the screen
            # need to remake the drawings of the points and path numbers
            path = np.array(Image.new("RGB", (width, height), (0,0,0)))
            # draw dots to mark each node at each node
            for i, v in enumerate(points):    # draws markers at every point
                cv2.circle(path, v, 1, (1, 1, 1), int(9*cqm))
                if i == 0 or i == matrix_length:
                    cv2.circle(path, v, 1, [202, 137, 65], int(6*cqm))
                else:
                    cv2.circle(path, v, 1, line_color, int(6*cqm))                
                cv2.circle(path, v, 1, (1, 1, 1), int(3*cqm))
            # draw numbers between paths
            text = np.array(Image.new("RGB", (width, height), (0,0,0)))
            for i, val in enumerate(adjacency_matrix):
                for j in range(i):
                    if adjacency_matrix[i][j] != 0:
                        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                            cv2.putText(text, f'{adjacency_matrix[i][j]}', (int((points[i][0] + points[j][0])/2) + dx, int((points[i][1] + points[j][1])/2) + dy), cv2.FONT_HERSHEY_SIMPLEX, 0.5*cqm, (255, 255, 255), int(2*cqm))
                        cv2.putText(text, f'{adjacency_matrix[i][j]}', (int((points[i][0] + points[j][0])/2), int((points[i][1] + points[j][1])/2)), cv2.FONT_HERSHEY_SIMPLEX, 0.5*cqm, (1, 1, 1), int(2*cqm))
            
            # redraw the remaining paths
            for i, v in enumerate(nodes_taken_ordered):  # i is index of nodes_taken_ordered, v is index of a node
                if i < len(nodes_taken_ordered) - 1:
                    node1 = nodes_taken_ordered[i]
                    node2 = nodes_taken_ordered[i+1]
                    cv2.line(path, (points[node1][0], points[node1][1]), (points[node2][0], points[node2][1]), line_color, line_thickness)




    if key == 27:               # 27 is ASCII for [ESC]
        key = 0
        break                   # HOLD [ESC] to end program
    
    ################################################################################################################################################
    ################################################################################################################################################
    ################################################################################################################################################

    # draw the solution to the optimization problem (hard coded)
    if complete == True:
        if matrix_length == 13:  # only display the score if the matrix is the right size
            
            if mode == 2: # SHORTEST PATH
                num_nodes = np.sum(nodes_taken) # counts how many nodes were taken in total
                path2 = np.array(Image.new("RGB", (width, height), (0,0,0)))
                path3 = np.array(Image.new("RGB", (width, height), (0,0,0)))
                
                # draws best path from each node touched to end
                lt = int(line_thickness + 6)    # line thickness
                lcc = line_color_check    # line color
                cv2.line(path3, (points[ 0][0], points[ 0][1]), (points[ 1][0], points[ 1][1]), lcc, lt)
                cv2.line(path3, (points[ 1][0], points[ 1][1]), (points[ 6][0], points[ 6][1]), lcc, lt)
                cv2.line(path3, (points[ 6][0], points[ 6][1]), (points[ 5][0], points[ 5][1]), lcc, lt)
                cv2.line(path3, (points[ 5][0], points[ 5][1]), (points[ 4][0], points[ 4][1]), lcc, lt)
                cv2.line(path3, (points[ 4][0], points[ 4][1]), (points[ 8][0], points[ 8][1]), lcc, lt)
                cv2.line(path3, (points[ 8][0], points[ 8][1]), (points[ 7][0], points[ 7][1]), lcc, lt)
                cv2.line(path3, (points[ 7][0], points[ 7][1]), (points[11][0], points[11][1]), lcc, lt)
                cv2.line(path3, (points[11][0], points[11][1]), (points[12][0], points[12][1]), lcc, lt)
                cv2.line(path3, (points[12][0], points[12][1]), (points[13][0], points[13][1]), lcc, lt)
                
                # caluculate and display score
                optimal_score = 45
                percent_score = int(100 * optimal_score / path_value)
                border_color = (1, 1, 1)
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    cv2.putText(path2, f'Score: {path_value}          {percent_score}%', (int(10*cqm) + 3*dx, int(50*cqm) + 3*dx), cv2.FONT_HERSHEY_TRIPLEX, 1.5*cqm, (1, 1, 1), int(2*cqm))
                        # technically dx and dy should be used above, but it looks cool if you just you dx for both
                cv2.putText(path2, f'Score: {path_value}          {percent_score}%', (int(10*cqm), int(50*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 1.5*cqm, (0, 200, 200), int(2*cqm))

                # display instruction text
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    cv2.putText(path2, 'Press [spacebar] to Reset', (int(25*cqm) + dx, int(height - 15*cqm) + dy), cv2.FONT_HERSHEY_TRIPLEX, 0.6*cqm, (1, 1, 1), int(1*cqm))
                cv2.putText(path2, 'Press [spacebar] to Reset', (int(25*cqm), int(height - 15*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 0.6*cqm, (150, 150, 150), int(1*cqm))
            
            else:  # LONGEST PATH
                num_nodes = np.sum(nodes_taken) # counts how many nodes were taken in total
                path2 = np.array(Image.new("RGB", (width, height), (0,0,0)))
                path3 = np.array(Image.new("RGB", (width, height), (0,0,0)))
                
                # draws best path from each node touched to end
                lt = int(line_thickness + 6)    # line thickness
                lcc = line_color_check    # line color
                cv2.line(path3, (points[ 0][0], points[ 0][1]), (points[ 3][0], points[ 3][1]), lcc, lt)
                cv2.line(path3, (points[ 3][0], points[ 3][1]), (points[ 1][0], points[ 1][1]), lcc, lt)
                cv2.line(path3, (points[ 1][0], points[ 1][1]), (points[ 2][0], points[ 2][1]), lcc, lt)
                cv2.line(path3, (points[ 2][0], points[ 2][1]), (points[ 6][0], points[ 6][1]), lcc, lt)
                cv2.line(path3, (points[ 6][0], points[ 6][1]), (points[ 5][0], points[ 5][1]), lcc, lt)
                cv2.line(path3, (points[ 5][0], points[ 5][1]), (points[ 9][0], points[ 9][1]), lcc, lt)
                cv2.line(path3, (points[ 9][0], points[ 9][1]), (points[ 8][0], points[ 8][1]), lcc, lt)
                cv2.line(path3, (points[ 8][0], points[ 8][1]), (points[ 7][0], points[ 7][1]), lcc, lt)
                cv2.line(path3, (points[ 7][0], points[ 7][1]), (points[12][0], points[12][1]), lcc, lt)
                cv2.line(path3, (points[12][0], points[12][1]), (points[10][0], points[10][1]), lcc, lt)
                cv2.line(path3, (points[10][0], points[10][1]), (points[13][0], points[13][1]), lcc, lt)
                
                # caluculate and display score
                optimal_score = 85
                percent_score = int(100 * path_value / optimal_score)
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    cv2.putText(path2, f'Score: {path_value}          {percent_score}%', (int(10*cqm) + 3*dx, int(50*cqm) + 3*dx), cv2.FONT_HERSHEY_TRIPLEX, 1.5*cqm, (1, 1, 1), int(2*cqm))
                        # technically dx and dy should be used above, but it looks cool if you just you dx for both
                cv2.putText(path2, f'Score: {path_value}          {percent_score}%', (int(10*cqm), int(50*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 1.5*cqm, (0, 200, 200), int(2*cqm))
            
                # display instrction text
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    cv2.putText(path2, 'Press [spacebar] to Reset', (int(25*cqm) + dx, int(height - 15*cqm) + dy), cv2.FONT_HERSHEY_TRIPLEX, 0.6*cqm, (1, 1, 1), int(1*cqm))
                cv2.putText(path2, 'Press [spacebar] to Reset', (int(25*cqm), int(height - 15*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 0.6*cqm, (150, 150, 150), int(1*cqm))

        else:  # when matrix is the wrong size for the hard coded solution
            path2 = np.array(Image.new("RGB", (width, height), (0,0,0)))  # path2 created so things don't break later on (but it's empty)
            # display instrction text
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                cv2.putText(path2, 'press [spacebar] to reset', (int(25*cqm) + dx, int(height - 15*cqm) + dy), cv2.FONT_HERSHEY_TRIPLEX, 0.6*cqm, (1, 1, 1), int(1*cqm))
            cv2.putText(path2, 'press [spacebar] to reset', (int(25*cqm), int(height - 15*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 0.6*cqm, (150, 150, 150), int(1*cqm))

    show_solutions = False  # solutions won't be displayed until it's True

    # loop draws the solution paths for every frame
    while complete == True:
        ret, frame = capture.read()
        if flip_camera:    # jmhere
            frame = cv2.flip(frame, 1)
        if frame.shape[1] / frame.shape[0] == 4 / 3:  # crops the video feed if it has 4:3 aspect ratio (so it doesn't stretch during resize)
            trim = int(frame.shape[0] * 0.125)  # trims 12.5% from each side of height to change into 16:9 aspect ratio
            frame = frame[trim:frame.shape[0] - trim, :]
        frame = cv2.resize(frame, (1280, 720), interpolation=cv2.INTER_AREA)
        
        if (not show_solutions):
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                cv2.putText(frame, 'Press [S] to Show Solutions', (int(25*cqm) + dx, int(height - 40*cqm) + dy), cv2.FONT_HERSHEY_TRIPLEX, 0.6*cqm, (0, 0, 0), int(1*cqm))
            cv2.putText(frame, 'Press [S] to Show Solutions', (int(25*cqm), int(height - 40*cqm)), cv2.FONT_HERSHEY_TRIPLEX, 0.6*cqm, (150, 150, 150), int(1*cqm))
        # overlay the path over the webcam feed
        if (show_solutions):
            # add path3 (solution paths)
            mask2 = cv2.cvtColor(path3, cv2.COLOR_BGR2GRAY)
            mask2 = cv2.threshold(mask2, 0, 255, cv2.THRESH_BINARY)[1]
            mask2inv = cv2.bitwise_not(mask2)
            pathfg = cv2.bitwise_and(path3, path3, mask=mask2)
            framebg = cv2.bitwise_and(frame, frame, mask=mask2inv)
            frame = cv2.add(framebg, pathfg)
        # add path2 (score and instructions)
        mask2 = cv2.cvtColor(path2, cv2.COLOR_BGR2GRAY)
        mask2 = cv2.threshold(mask2, 0, 255, cv2.THRESH_BINARY)[1]
        mask2inv = cv2.bitwise_not(mask2)
        pathfg = cv2.bitwise_and(path2, path2, mask=mask2) # extract foreground
        framebg = cv2.bitwise_and(frame, frame, mask=mask2inv) # extract background
        frame = cv2.add(framebg, pathfg) # combine foreground and background
        # original path
        mask2 = cv2.cvtColor(path, cv2.COLOR_BGR2GRAY)
        mask2 = cv2.threshold(mask2, 0, 255, cv2.THRESH_BINARY)[1]
        mask2inv = cv2.bitwise_not(mask2)
        pathfg = cv2.bitwise_and(path, path, mask=mask2) # extract foreground
        framebg = cv2.bitwise_and(frame, frame, mask=mask2inv) # extract background
        frame = cv2.add(framebg, pathfg) # combine foreground and background
        # numbers between nodes
        mask2 = cv2.cvtColor(text, cv2.COLOR_BGR2GRAY)
        mask2 = cv2.threshold(mask2, 0, 255, cv2.THRESH_BINARY)[1]
        mask2inv = cv2.bitwise_not(mask2)
        pathfg = cv2.bitwise_and(text, text, mask=mask2) # extract foreground
        framebg = cv2.bitwise_and(frame, frame, mask=mask2inv) # extract background
        frame = cv2.add(framebg, pathfg) # combine foreground and background
        
        cv2.imshow('Supply Chain Maze', frame)
        
        key = cv2.waitKey(10) & 0xFF
        
        if key == 115 or key == 83:       # 115 is ASCII for [s], 83 for [S]
            show_solutions = True         # displays solutions if [s] or [S] is pressed

        if key == 27:               # 27 is ASCII for [ESC]
            break                   # ends loop when [ESC] is pressed

        if key == 32:               # 32 is ASCII for [spacebar]
            mode = 0
            break                   # ends loop when [spacebar] is pressed

    if key == 27:               # 27 is ASCII for [ESC]
        key = 0
        mode = 0
        break                   # HOLD [ESC] to end program

capture.release()
cv2.destroyAllWindows()  # closes window, only reaches here when [ESC] is pressed